# 파서학개론

## PDF란?

### **PDF 기본 요소**
**PDF의 가장 기본 요소 = 객체**

두 종류의 객체가 존재함.
1. 직접 객체
2. 간접 객체

**1. 직접 객체**
```
true false                       % Boolean
123 -98 34.5                     % Number
(Hello)                          % Literal string
<48656C6C6F>                     % Hexadecimal string
/Type /Catalog                   % Name object
[0 0 612 792]                    % Array
<< /Type /Page /Parent 2 0 R >>  % Dictionary
stream ... endstream             % Stream object
null                             % Null object
```

**2. 간접 객체**
```
12 0 obj ... endobj              % 간접 객체 선언
12 0 R                           % 간접 객체 호출
```

*\+ 간접 객체 식별 방법*
```
<object number> <generation number> ...
```

---

### **파일 구조**

#### **PDF = Header + Body + Cross-reference table + Trailer**

Header : 파일의 첫 줄. PDF 규격의 버전을 식별함.

Body : 간접 객체들 + 문서 요소 표현 + 객체 스트림

Cross-reference table : 간접 객체들이 파일의 어느 위치에 있는지(바이트 오프셋)에 대한 정보를 담고 있는 테이블.

Trailer : PDF 리더기가 문서의 구조를 파악하기 위해 가장 먼저 참고하는 영역.

---

#### PDF는 뒤에서 앞으로 해석!!!

---

### **[1] Header**
시작이 되는 부분

예시
```
%PDF-1.7
%도도
```

`%`는 PDF에서 주석을 의미. 
첫 번째 주석은 PDf 버전을 의미
두 번째 주석은 해당 파일이 바이너리 파일이라는 의미 (128 이상의 바이트를 최소 4개 이상 넣음)

### **[2] Trailer**

**Trailer 역할**
1. Cross-reference table의 위치 제공
2. Body의 특정 특수 객체들의 위치 제공
3. 파일의 끝을 알리는 %%EOF 마커
4. 상호 참조 테이블이 시작되는 바이트 오프셋 위치(startxref)
5. 문서 계층 구조의 최상위 객체인 카탈로그 딕셔너리(Root) 정보

**Trailer 예시**
```
trailer
<<
    /Size 117
    /Root 1 0 R
    /Info 14 0 R
    /ID[
        <9B8EA23639C1DA4FA67B9B17AEFF023D>
        <9B8EA23639C1DA4FA67B9B17AEFF023D>
    ] 
>>
startxref
27593
%%EOF
```

Trailer는 PDF 파일을 수정하면 추가될 수 있다. 그런 경우 추가된 객체와 /Prev 와 /XReStm 등으로 추가 정보를 표현한다.

예시
```
trailer
<<
    /Size 117
    /Root 1 0 R
    /Info 14 0 R
    /ID[
        <9B8EA23639C1DA4FA67B9B17AEFF023D>
        <9B8EA23639C1DA4FA67B9B17AEFF023D>
    ] 
>>
startxref
27593
%%EOF

xref
0 0
trailer
<<
/Size 117
/Root 1 0 R
/Info 14 0 R
/ID[
    <9B8EA23639C1DA4FA67B9B17AEFF023D>
    <9B8EA23639C1DA4FA67B9B17AEFF023D>
] 
/Prev 27593
/XRefStm 27121
>>
startxref
30092
%%EOF
```

---

### **[3] Cross-reference table**

줄여서 xref 라고 PDF에 표기함.

trailer에 존재하는 startxref가 xref의 위치를 알려줌.

xref 태그부터 trailer 태그 전까지가 모두 Cross-reference table 영역.

예시
```
xref
0 6
0000000003 65535 f
0000000017 00000 n
0000000081 00000 n
0000000000 00007 f
0000000331 00000 n
0000000409 00000 n
```

의미 해석

```
xref = cross-reference table 시작
0 6 = 0부터 시작하는 객체가 6개 있다.
0000000003 65535 f = 0번 객체
0000000017 00000 n = 1번 객체
0000000081 00000 n = 2번 객체
0000000000 00007 f = 3번 객체
0000000331 00000 n = 4번 객체
0000000409 00000 n = 5번 객체
```

**객체 사용 여부**
```
n = in-use (사용 중) | f = free (사용 안함)
```



**앞의 두 숫자의 의미**
1. n인 경우
```
0000000017 00000 n

<byte offset> <generation> n
```

해석 >> 17 바이트 위치에 해당 객체가 시작되고, 이 객체의 세대(버전)는 0번 이고, 사용되고 있는 객체이다.

2. f인 경우
```
0000000003 65535 f

<next free obj num> <generation> f
```
해석 >> 다음 free 객체의 번호는 3 이고, 더 이상 재사용 금지! (65535는 PDF generation number의 최대값)

---

#### **(실습) PDF에서 Trailer와 Cross-reference table 확인해보기**

In [ ]:
# PDF 선택
pdf_path = "파서학개론.pdf"

In [ ]:
# PDF를 byte로 읽기
raw_pdf = None

with open(pdf_path, "rb") as f:
    raw_pdf = f.read()

In [ ]:
# header 100 바이트만 출력
print(raw_pdf[:100])

In [ ]:
# tail 100 바이트만 출력
print(raw_pdf[-100:])

In [ ]:
# 예쁘게 보기 위해서 latin-1 디코딩 적용
print(raw_pdf[:100].decode('latin-1'))

In [ ]:
print(raw_pdf[-100:].decode('latin-1'))

In [ ]:
# trailer의 구성요소 중 하나인 %%EOF 확인 가능
print(raw_pdf[-5:].decode("latin-1"))

In [ ]:
# # startxref가 가리키는 byte offset에서 끝까지(xref + trailer) 출력
# import re

# m = list(re.finditer(rb"startxref\s+(\d+)\s+%%EOF", raw_pdf, flags=re.DOTALL))[-1]
# latest_xref_offset = int(m.group(1))

# print(f"latest startxref offset: {latest_xref_offset}")
# print(raw_pdf[latest_xref_offset:].decode("latin-1", errors="replace"))

In [ ]:
# startxref를 따라 30092 byteoffset에서 끝까지(xref + trailer) 출력
print(raw_pdf[30092:].decode("latin-1"))

In [ ]:
# # 깔끔하게 보기 위해서 / 가 나오면 개행 진행.
# xref_trailer1 = raw_pdf[latest_xref_offset:].decode("latin-1", errors="replace")
# xref_trailer1 = xref_trailer1.replace("/", "\n/")
# print(xref_trailer1)

In [ ]:
# 깔끔하게 보기 위해서 / 가 나오면 개행 진행.
xref_trailer1 = raw_pdf[30092:].decode("latin-1")
xref_trailer1 = xref_trailer1.replace('/', '\n/')
print(xref_trailer1)

In [ ]:
# # /Prev를 통해 이전 xref + trailer 확인
# prev_match = re.search(r"/Prev\s+(\d+)", xref_trailer1)
# prev_xref_offset = int(prev_match.group(1)) if prev_match else None

# if prev_xref_offset is None:
#     print("/Prev가 없습니다. incremental update가 없는 PDF일 수 있습니다.")
# else:
#     print(f"previous xref offset: {prev_xref_offset}")
#     xref_trailer2 = raw_pdf[prev_xref_offset:].decode("latin-1", errors="replace")
#     xref_trailer2 = xref_trailer2.replace("/", "\n/")
#     print(xref_trailer2)

In [ ]:
# /Prev를 통해 이전 xref + trailer 확인
xref_trailer2 = raw_pdf[27593:].decode("latin-1")
xref_trailer2 = xref_trailer2.replace('/', '\n/')
print(xref_trailer2)

In [ ]:
# # 이전 xref + trailer만 출력
# # /Prev가 없는 PDF라면 최신 xref + trailer를 그대로 사용합니다.
# if prev_xref_offset is None:
#     print("이전 xref가 없습니다. 최신 xref 섹션을 사용합니다.")
#     xref_trailer2_only = raw_pdf[latest_xref_offset:].decode("latin-1", errors="replace")
# else:
#     xref_trailer2_only = raw_pdf[prev_xref_offset:latest_xref_offset].decode("latin-1", errors="replace")

# xref_trailer2_only = xref_trailer2_only.replace("/", "\n/")
# print(xref_trailer2_only)

In [ ]:
# 이전 xref + trailer만 출력
# /Prev를 통해 이전 xref + trailer 확인
xref_trailer2_only = raw_pdf[27593:30092].decode("latin-1")
xref_trailer2_only = xref_trailer2_only.replace('/', '\n/')
print(xref_trailer2_only)

Xref와 Trailer 분리

In [ ]:
xref_trailer2_only.index("trailer")

In [ ]:
xref_only = xref_trailer2_only[:2353]

print(xref_only)

In [ ]:
trailer_only = xref_trailer2_only[2353:]
print(trailer_only)

Xref에서 Object가 117개 인 것 확인 가능. 해당 객체들에 접근하기 쉽게 딕셔너리로 저장하기 (Parsing)

In [ ]:
# xref_list = xref_only.splitlines()

# print(xref_list)

In [ ]:
xref_list = xref_only.split('\r\n')

print(xref_list)

In [8]:
# # 앞부분에 위치한 `xref`와 `객체 갯수`에 대한 정보와 빈 문자열 제거
# xref_list_clean = [line.strip() for line in xref_list if line.strip()]
# obj_only = xref_list_clean[2:]

In [ ]:
# 앞부분에 위치한 `xref`와 `객체 갯수`에 대한 정보와 끝에 존재하는 `빈 문자열` 제거
obj_only = xref_list[2:-1]

In [ ]:
# obj_dict = {}

# for i, obj in enumerate(obj_only):
#     obj_dict[f"obj{i}"] = obj

# print(obj_dict)

In [ ]:
obj_dict = {}

i = 0
for obj in obj_only:
    obj_dict[f"obj{i}"] = obj
    i += 1

print(obj_dict)

In [ ]:
# 보기 편하게 10개만 출력

i = 0
for key, value in obj_dict.items():
    if i == 10: break
    
    print(f"{key}: {value}")

    i += 1

In [ ]:
print(len(obj_dict))

---

### **[4] Body**



기본적으로 Body는 간접 객체들의 묶음으로 보면 됨.

Header가 파일의 종류를 알려주고, Trailer와 Xref가 객체의 위치를 알려준다면, Body는 실제 내용이 들어 있는 구간임.

**Body에서 자주 만나는 객체**
1. Catalog : 문서의 시작점. Trailer의 `/Root`가 여기로 감.
2. Pages : 페이지들을 묶는 트리 노드.
3. Page : 한 장의 페이지. `/MediaBox`, `/Resources`, `/Contents`가 중요함.
4. Resources : 페이지 안에서 쓰는 글꼴, 이미지, 색상 공간 등의 이름표.
5. Stream : 압축된 데이터 덩어리. 페이지 명령어, 이미지, 폰트 파일 등이 들어감.
6. Object Stream : 여러 객체를 압축해서 한 stream 안에 넣은 형태.

#### Body를 따라가는 순서

PDF는 앞에서부터 차례대로 읽기보다, 뒤에서 얻은 단서를 따라 들어가는 편이 자연스러움.

```text
startxref
  -> xref
  -> trailer
  -> /Root
  -> Catalog
  -> /Pages
  -> Page
  -> /Resources, /Contents
```

#### 객체를 볼 때의 기준

간접 객체는 보통 다음 모양임.

```pdf
12 0 obj
<< ... >>
stream
...
endstream
endobj
```

`12 0 obj`는 객체 번호와 generation 번호임.  
`12 0 R`은 해당 객체를 참조한다는 뜻임.

---

### PDF 해석 1단계 (Parsing)

PDF에서 어디가 Header, Body, Xref, Trailer인지 구조에 맞게 구분한다.

### PDF 해석 2단계 (Decoding)

Stream 데이터를 이해할 수 있게 디코딩을 진행한다.

### PDF 해석 3단계 (Parsing)

### PDF 해석 4단계 (Resource)

### PDF 해석 5단계 (Rendering)